In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import math

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

df = load_dataset('크루즈입항정보')

In [3]:
frequency_source = df.copy()
frequency_source.columns = frequency_source.columns.str.strip()

place_col = "계선장소.2"

if place_col not in frequency_source.columns:
    raise ValueError("'계선장소.2' 컬럼을 확인해주세요.")

frequency_data = frequency_source[
    (frequency_source["입출"] == "입항") &
    (frequency_source["선박용도"] == "크루즈선")
    ].copy()

frequency_data["입항일시"] = pd.to_datetime(
    frequency_data["입항일시"],
    errors="coerce"
)

frequency_data = frequency_data[
    (frequency_data["입항일시"] >= "2013-01-01") &
    (frequency_data["입항일시"] < "2026-10-17")
    ].copy()

if frequency_data.empty:
    raise ValueError("분석 기간에 해당하는 데이터가 없습니다.")

frequency_data["계선장소"] = (
    frequency_data[place_col]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("계선장소 미상")
)

frequency_table = (
    frequency_data.groupby("계선장소")
    .size()
    .reset_index(name="입항빈도")
    .sort_values(
        ["입항빈도", "계선장소"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

total_count = frequency_table["입항빈도"].sum()

chart_data = frequency_table.head(5).copy()
chart_data["기타여부"] = False

remaining = frequency_table.iloc[5:].copy()

if not remaining.empty:
    other = pd.DataFrame({
        "계선장소": ["기타"],
        "입항빈도": [remaining["입항빈도"].sum()],
        "기타여부": [True]
    })

    chart_data = pd.concat(
        [chart_data, other],
        ignore_index=True
    )

chart_data["입항비중"] = (
        chart_data["입항빈도"] / total_count * 100
)

chart_data["장소표시"] = (
    chart_data["계선장소"]
    .str.replace("(", "<br>(", regex=False)
)

max_count = frequency_table["입항빈도"].max()

# 최댓값: 주황 / 나머지: 파랑 / 기타: 회색
colors = [
    "#A6AFBA" if is_other
    else "#E67E22" if count == max_count
    else "#3478C0"
    for count, is_other in zip(
        chart_data["입항빈도"],
        chart_data["기타여부"]
    )
]

labels = [
    f"{count:,.0f}회 ({share:.1f}%)"
    for count, share in zip(
        chart_data["입항빈도"],
        chart_data["입항비중"]
    )
]

fig_berth_frequency = go.Figure()

fig_berth_frequency.add_trace(
    go.Bar(
        x=chart_data["입항빈도"],
        y=chart_data["장소표시"],
        orientation="h",
        marker_color=colors,
        marker_line_width=0,
        text=labels,
        textposition="outside",
        textfont=dict(size=17, color="#243247"),
        cliponaxis=False,
        customdata=chart_data[
            ["계선장소", "입항비중"]
        ].to_numpy(),
        hovertemplate=(
            "<b>%{customdata[0]}</b>"
            "<br>입항 빈도: %{x:,.0f}회"
            "<br>전체 입항 중 비중: %{customdata[1]:.1f}%"
            "<extra></extra>"
        )
    )
)

fig_berth_frequency.update_layout(
    title=dict(
        text=(
            "<b>부산항 크루즈 계선장소별 입항 빈도</b>"
            "<br>"
            "<span style='font-size:16px;color:#657487'>"
            "2013.01.01~2026.10.16 · TOP 5 및 기타"
            "</span>"
        ),
        x=0.5,
        y=0.96,
        xanchor="center",
        font=dict(size=23)
    ),
    template="plotly_white",
    width=1200,
    height=675,
    paper_bgcolor="white",
    plot_bgcolor="white",
    font=dict(
        family="Malgun Gothic, Arial, sans-serif",
        size=16,
        color="#243247"
    ),
    margin=dict(l=260, r=70, t=125, b=125),
    bargap=0.35,
    showlegend=False
)

fig_berth_frequency.update_yaxes(
    title=None,
    categoryorder="array",
    categoryarray=chart_data["장소표시"].tolist(),
    autorange="reversed",
    showgrid=False,
    automargin=True,
    tickfont=dict(size=15)
)

fig_berth_frequency.update_xaxes(
    title="입항 빈도 (회)",
    range=[0, chart_data["입항빈도"].max() * 1.40],
    tickformat=",d",
    gridcolor="#E9EDF2",
    zeroline=False
)

fig_berth_frequency.add_annotation(
    x=0,
    y=-0.20,
    xref="paper",
    yref="paper",
    text=(
        "※ 입항 빈도는 분석 기간의 입항 횟수이며, 괄호는 전체 입항 중 비중입니다."
        "<br>※ 한글 장소명 기준으로 선석별 구분. 기타는 상위 5개를 제외한 장소의 합계입니다."
    ),
    showarrow=False,
    xanchor="left",
    align="left",
    font=dict(size=12, color="#657487")
)

fig_berth_frequency.show(
    config={
        "displaylogo": False,
        "toImageButtonOptions": {
            "format": "png",
            "filename": "PPT_계선장소별_입항빈도_TOP5_기타",
            "width": 1200,
            "height": 675,
            "scale": 3
        }
    }
)